<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 생성 모델 · 01. VAE

## Auto-Encoding Variational Bayes

- **원 논문:** [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114)
- **저자 / 발표:** Diederik P. Kingma and Max Welling · ICLR (2014)
- **난이도 / 예상 시간:** 중급 · 약 55분
- **선수 지식:** Gaussian 분포, KL divergence, PyTorch autograd
- **로컬 학습 데이터:** `data/field_curriculum/generative_samples.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 2차원 로컬 점에서 reparameterization, 닫힌형 KL, SGVB ELBO를 구현한다.

**축소하거나 생략한 부분:** 논문은 MNIST/Frey Face에 stochastic variational inference를 적용한다. 이 축소판은 동일한 ELBO를 2차원 mixture에 적용한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2.2, Eq. (1)–(3) | evidence lower bound | reconstruction 기대값과 posterior-prior KL로 log evidence의 하한을 구성한다. |
| §2.3, Eq. (4)–(6), Algorithm 1 | SGVB estimator | 재매개변수화한 표본으로 encoder까지 역전파한다. |
| §2.4 and §3, Eq. (9)–(10) | Gaussian posterior | z=mu+sigma*epsilon 및 diagonal Gaussian KL을 구현한다. |

## 핵심 재현

        Eq. (3)의 ELBO와 Eq. (9)–(10)의 Gaussian 특수화를 사용합니다. 고정된 로컬 mixture에서
        $z=\mu+\exp(\frac12\log\sigma^2)\epsilon$ 경로가 미분 가능한지 검증합니다.

In [ ]:
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(311)
np.random.seed(311)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# These teaching batches are intentionally tiny; GPU launch/transfer
# overhead can dominate, so use AI_LAB_DEVICE=cpu for minimum wall time.

DATA_PATH = Path("data/field_curriculum/generative_samples.npz")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
raw = np.load(DATA_PATH)
expected = {"points", "point_labels", "images", "image_labels", "domain_a", "domain_b"}
assert expected.issubset(raw.files), f"NPZ 키 불일치: {sorted(raw.files)}"
points = torch.tensor(raw["points"], dtype=torch.float32)
point_labels = torch.tensor(raw["point_labels"], dtype=torch.long)
images = torch.tensor(raw["images"], dtype=torch.float32)
image_labels = torch.tensor(raw["image_labels"], dtype=torch.long)
domain_a = torch.tensor(raw["domain_a"], dtype=torch.float32)
domain_b = torch.tensor(raw["domain_b"], dtype=torch.float32)
points, point_labels, images, image_labels, domain_a, domain_b = ACCELERATOR.move(
    points, point_labels, images, image_labels, domain_a, domain_b
)


# Seeded random draws stay on CPU so CUDA, MPS, and CPU follow the same
# sampling stream; complete tensors are then transferred to the lab device.
def randn_device(size, *, generator):
    return torch.randn(size, generator=generator).to(DEVICE)


def rand_device(size, *, generator):
    return torch.rand(size, generator=generator).to(DEVICE)


def randint_device(high, size, *, generator):
    return torch.randint(high, size, generator=generator).to(DEVICE)


if float(images.min()) < 0.0:
    images_01 = ((images + 1.0) / 2.0).clamp(0.0, 1.0)
else:
    images_01 = images.clamp(0.0, 1.0)
assert points.shape == (600, 2) and images.shape == (300, 1, 8, 8)
assert domain_a.shape == domain_b.shape == (600, 2)
print(ACCELERATOR.summary())
print("local data:", tuple(points.shape), tuple(images.shape), tuple(domain_a.shape))

## 포트폴리오 구현 설계: 수식 → 책임 → 검증

논문의 핵심 목적을 실행 가능한 객체의 `forward`, `loss`, `update`, `evaluate`로
나눕니다. 아래 식이 클래스 내부에서 생략되지 않고 어떤 tensor 연산이 되는지
메서드별로 추적해 보세요.

$$
-\mathcal L_{ELBO}=\mathbb E_{q_\phi(z|x)}[-\log p_\theta(x|z)]+\beta D_{KL}(q_\phi(z|x)\Vert p(z))
$$

- **기호와 역할:** $q_\phi$는 encoder posterior, $p_\theta$는 decoder likelihood, $z=\mu+\sigma\epsilon$은 재매개화 표본입니다.
- **shape 계약:** `point x [N, 2] → (mu, logvar) [N, 2] → z [N, 2] → recon [N, 2]`
- **논문 위치:** `§2.2, Eq. (1)–(3)`의 **evidence lower bound**
- **코드 Task:** 재매개화, diagonal Gaussian KL, negative ELBO와 deterministic 평가를 작성합니다.
- **학습·평가 흐름:** encode → rsample → decode → reconstruction+KL update → posterior-mean 평가
- **원문 대비 한계:** 논문은 MNIST/Frey Face에 stochastic variational inference를 적용한다. 이 축소판은 동일한 ELBO를 2차원 mixture에 적용한다.
- **구현 이유:** 수식의 tensor 경계와 optimizer 책임을 클래스 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 교육용 편의 함수가 핵심 계산을 대신하지 않도록 loss와 update 순서를
메서드 본문에 드러냅니다. 실습본에서는 같은 공개 API를 유지한 채 TODO를 채우세요.

In [ ]:
class VariationalAutoencoderLab(nn.Module):
    """VAE의 핵심 학습·평가 경로. 입출력 계약: point x [N, 2] → (mu, logvar) [N, 2] → z [N, 2] →
    recon [N, 2]."""

    def __init__(self, network, optimizer, beta=0.08):
        """구성 요소와 optimizer를 저장한다. 학습은 update에서만 수행한다."""
        super().__init__()
        self.network = network
        self.optimizer = optimizer
        self.beta = beta

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        return {"data_dim": 2, "latent_dim": 2, "beta": self.beta}

    def forward(self, batch, deterministic=False):
        """논문 순전파를 계산한다. shape 계약: point x [N, 2] → (mu, logvar) [N, 2] → z [N, 2] →
        recon [N, 2]."""
        mu, logvar = self.network.encode(batch)
        standard_deviation = torch.exp(0.5 * logvar)
        epsilon = torch.zeros_like(mu) if deterministic else torch.randn_like(mu)
        latent = mu + standard_deviation * epsilon
        reconstruction = self.network.dec(latent)
        return (reconstruction, mu, logvar)

    def loss(self, batch):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        reconstruction, mu, logvar = self.forward(batch)
        reconstruction_term = (reconstruction - batch).square().sum(dim=1)
        kl_term = -0.5 * (1 + logvar - mu.square() - logvar.exp()).sum(dim=1)
        return (reconstruction_term + self.beta * kl_term).mean()

    def update(self, batch):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        objective = self.loss(batch)
        self.optimizer.zero_grad()
        objective.backward()
        self.optimizer.step()
        return float(objective.detach())

    def evaluate(self, batch):
        """no_grad 경로에서 논문과 연결된 정량 지표 dict를 반환한다."""
        with torch.no_grad():
            reconstruction, mu, logvar = self.forward(batch, deterministic=True)
            mse = F.mse_loss(reconstruction, batch)
            kl = -0.5 * (1 + logvar - mu.square() - logvar.exp()).sum(dim=1)
        return {"mse": float(mse), "mean_kl": float(kl.mean())}

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `§2.4 and §3, Eq. (9)–(10)` — **Gaussian posterior**
- **구현 이유:** z=mu+sigma*epsilon 및 diagonal Gaussian KL을 구현한다.
- **읽을 코드:** 아래 `implementation` 셀에서 중간 tensor의 shape, gradient가 흐르는
  경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
def reparameterize(mu, logvar, eps=None):
    std = torch.exp(0.5 * logvar)
    return mu + std * (torch.randn_like(std) if eps is None else eps)


def gaussian_kl(mu, logvar):
    return -0.5 * (1 + logvar - mu.square() - logvar.exp()).sum(1)


mu0 = torch.zeros(4, 2, requires_grad=True)
lv0 = torch.zeros_like(mu0)
z0 = reparameterize(mu0, lv0, torch.zeros_like(mu0))
assert torch.allclose(z0, mu0) and torch.allclose(gaussian_kl(mu0, lv0), torch.zeros(4))
z0.sum().backward()
assert mu0.grad is not None

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `§2.3, Eq. (4)–(6), Algorithm 1` — **SGVB estimator**
- **구현 이유:** 재매개변수화한 표본으로 encoder까지 역전파한다.
- **읽을 코드:** 아래 `training` 셀에서 중간 tensor의 shape, gradient가 흐르는
  경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
class TinyVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(2, 16), nn.Tanh())
        self.mu, self.lv = nn.Linear(16, 2), nn.Linear(16, 2)
        self.dec = nn.Sequential(nn.Linear(2, 16), nn.Tanh(), nn.Linear(16, 2))

    def encode(self, x):
        h = self.enc(x)
        return self.mu(h), self.lv(h).clamp(-6, 6)

    def forward(self, x, deterministic=False):
        mu, lv = self.encode(x)
        eps = torch.zeros_like(mu) if deterministic else None
        return self.dec(reparameterize(mu, lv, eps)), mu, lv


def vae_loss(model, x, deterministic=False):
    recon, mu, lv = model(x, deterministic)
    rec = (recon - x).square().sum(1)
    return (rec + 0.08 * gaussian_kl(mu, lv)).mean(), rec.mean()


model = TinyVAE().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.015)
history = []
x_train = points[:480]
with torch.no_grad():
    initial = float(vae_loss(model, x_train, True)[0])

portfolio_model = VariationalAutoencoderLab(model, optimizer)
initial_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in model.parameters()]
)
for _ in range(150):
    history.append(portfolio_model.update(x_train))
final_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in model.parameters()]
)
parameter_delta = float((final_parameters - initial_parameters).abs().sum())
assert history[-1] < initial and parameter_delta > 0.0
assert np.isfinite(history).all()

### 단계 4. 평가·시각화

        - **논문의 어느 부분인가:** `§2.2, Eq. (1)–(3)` — **evidence lower bound**
        - **구현 이유:** reconstruction 기대값과 posterior-prior KL로 log evidence의 하한을 구성한다.
        - **읽을 코드:** 아래 `evaluation` 셀에서 중간 tensor의 shape, gradient가 흐르는
          경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

        실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
        주석으로 적은 뒤 실습하세요.
        - **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

- **평가 범위:** 이 값은 작은 학습 batch의 메커니즘 진단이며, held-out 생성 품질이나 원 논문의 benchmark 성능으로 해석하지 않습니다.

In [ ]:
portfolio_metrics = portfolio_model.evaluate(points[480:])
assert portfolio_metrics

with torch.no_grad():
    recon, mu, lv = model(points[480:], True)
    mse = float(F.mse_loss(recon, points[480:]))
    mean_kl = float(gaussian_kl(mu, lv).mean())
print(f"reconstruction MSE={mse:.4f}, KL={mean_kl:.4f}")
assert math.isfinite(mse) and math.isfinite(mean_kl) and mean_kl >= 0
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].plot(history)
axes[0].set_title("negative ELBO")
mu_cpu, labels_cpu = mu.detach().cpu(), point_labels[480:].detach().cpu()
points_cpu, recon_cpu = points[480:].detach().cpu(), recon.detach().cpu()
axes[1].scatter(mu_cpu[:, 0], mu_cpu[:, 1], c=labels_cpu, s=10)
axes[1].set_title("q mean")
axes[2].scatter(points_cpu[:, 0], points_cpu[:, 1], alpha=0.35, label="data")
axes[2].scatter(recon_cpu[:, 0], recon_cpu[:, 1], s=10, label="recon")
axes[2].legend()
fig.tight_layout()
plt.show()

posterior mean을 사용하면 Monte Carlo 표본 잡음을 제거해 모델 자체의 평균 reconstruction을 비교할 수 있습니다.

### 단계 5. 통합 클래스가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `§2.2, Eq. (1)–(3)`의 **evidence lower bound**
- **핵심 식:**

  $$
  -\mathcal L_{ELBO}=\mathbb E_{q_\phi(z|x)}[-\log p_\theta(x|z)]+\beta D_{KL}(q_\phi(z|x)\Vert p(z))
  $$

- **입출력 shape:** `point x [N, 2] → (mu, logvar) [N, 2] → z [N, 2] → recon [N, 2]`
- **구현 이유:** 앞 셀은 수식을 작은 연산으로 분해해 확인하고,
  이 셀은 같은 구성 요소를 `VariationalAutoencoderLab`에 주입해
  최종 학습·평가 경로를 하나로 묶습니다. reconstruction 기대값과 posterior-prior KL로 log evidence의 하한을 구성한다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 논문 대응 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

In [ ]:
config_contract = portfolio_model.config()
numeric_metrics = [
    float(value)
    for value in portfolio_metrics.values()
    if isinstance(value, (int, float))
]
assert config_contract and len(history) > 0
assert np.isfinite(np.asarray(history, dtype=float)).all()
assert parameter_delta > 0.0
assert numeric_metrics and np.isfinite(numeric_metrics).all()
print(
    {
        "class": type(portfolio_model).__name__,
        "steps": len(history),
        "parameter_delta": parameter_delta,
        "metrics": portfolio_metrics,
    }
)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 논문은 MNIST/Frey Face에 stochastic variational inference를 적용한다. 이 축소판은 동일한 ELBO를 2차원 mixture에 적용한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.